# auditPaper_assist — dense 임베딩 + Qdrant 적재 (Colab GPU)

저장소 `main`에서 입력을 직접 확보해 **BAAI/bge-m3** dense를 계산하고, 그대로 `--stage upsert`까지 수행한다.

- 사전 조건: 로컬의 코퍼스 수정이 **push 완료**된 상태여야 함 (셀 2가 신선도를 assert)
- 런타임 유형: **GPU** (T4/L4)
- 마지막 셀이 내려주는 `manifest.json`·`build_report.md`·`embed_meta.json`을 로컬 `index/`에 배치해 커밋할 것

In [ ]:
# 셀 1 — GPU 확인 + 의존성 설치
!nvidia-smi -L
!pip -q install FlagEmbedding qdrant-client kiwipiepy

In [ ]:
# 셀 2 — 저장소 확보 + 코퍼스 커밋 SHA 기록 + 입력 신선도 검증
import json, os, urllib.request

REPO = "worldoftoddl/auditPaper_assist"
sha = json.load(urllib.request.urlopen(f"https://api.github.com/repos/{REPO}/commits/main"))["sha"]
os.environ["CORPUS_COMMIT"] = sha
print("코퍼스 커밋:", sha)

!wget -q https://codeload.github.com/{REPO}/tar.gz/refs/heads/main -O repo.tgz
!rm -rf repo && mkdir -p repo && tar xzf repo.tgz -C repo --strip-components=1
assert os.path.isfile("repo/scripts/build_index.py"), "저장소 확보 실패"

rows = [json.loads(l) for l in open("repo/index/embed_input.jsonl", encoding="utf-8")]
EXPECTED = 10063
assert len(rows) == EXPECTED, f"행 수 {len(rows)} != {EXPECTED}"
# 신선도: FRMK-1/ASSR-3000 제목 오염 수정(로컬 커밋)이 반영된 입력인지 확인
frmk = next(r["text"] for r in rows if r["cid"].startswith("KSA::FRMK-1::"))
assert frmk.startswith("감사기준 FRMK-1 인증업무개념체계"), f"구판 입력(제목 미수정): {frmk[:40]} — push 후 재실행"
print("입력 확보·신선도 OK:", len(rows), "행")

In [ ]:
# 셀 3 — bge-m3 dense 인코딩 (T4 기준 5~10분) + 정규화 보증
import numpy as np, torch
from FlagEmbedding import BGEM3FlagModel

assert torch.cuda.is_available(), "GPU 런타임 아님"
MODEL = "BAAI/bge-m3"
model = BGEM3FlagModel(MODEL, use_fp16=True)
out = model.encode([r["text"] for r in rows], batch_size=64, max_length=8192,
                   return_dense=True, return_sparse=False, return_colbert_vecs=False)
emb = np.asarray(out["dense_vecs"], dtype="float32")
assert emb.shape == (EXPECTED, 1024), emb.shape
assert not np.isnan(emb).any(), "NaN 존재"
n = np.linalg.norm(emb, axis=1)
print("norm range:", n.min(), n.max())
if not np.allclose(n, 1.0, atol=1e-3):
    print("[경고] 노름 이탈 → 재정규화")
    emb = emb / n[:, None]

In [ ]:
# 셀 4 — 산출물 + 산출물 매니페스트(embed_meta.json) 저장 → repo/index/ 배치
import hashlib, datetime, FlagEmbedding, transformers

np.save("embeddings.npy", emb)
cids = [r["cid"] for r in rows]
json.dump(cids, open("cids.json", "w", encoding="utf-8"), ensure_ascii=False)

sha256 = lambda p: hashlib.sha256(open(p, "rb").read()).hexdigest()
meta = {
    "embedding_device": f"colab-{torch.cuda.get_device_name(0)}-fp16",
    "embedding_runtime": (f"FlagEmbedding {FlagEmbedding.__version__}, "
                          f"torch {torch.__version__}, transformers {transformers.__version__}"),
    "model": MODEL,
    "rows": len(rows), "dim": int(emb.shape[1]), "dtype": str(emb.dtype),
    "corpus_commit": os.environ["CORPUS_COMMIT"],
    "input_sha256": sha256("repo/index/embed_input.jsonl"),
    "embeddings_sha256": sha256("embeddings.npy"),
    "cids_sha256": sha256("cids.json"),
    "generated_at": datetime.datetime.now(datetime.timezone.utc).isoformat(),
}
json.dump(meta, open("embed_meta.json", "w", encoding="utf-8"), ensure_ascii=False, indent=2)
!cp embeddings.npy cids.json embed_meta.json repo/index/
print(json.dumps(meta, ensure_ascii=False, indent=2))

In [ ]:
# 셀 5 — Qdrant 접속 정보 입력 + 업서트·점검·스모크·매니페스트 (전 과정)
from getpass import getpass
os.environ["QDRANT_URL"] = getpass("QDRANT_URL: ").rstrip("/")
os.environ["QDRANT_API_KEY"] = getpass("QDRANT_API_KEY: ")

%cd /content/repo
!python scripts/build_index.py --stage upsert --embeddings index/embeddings.npy --cids index/cids.json
%cd /content

In [ ]:
# 셀 6 — 산출물 회수: 로컬 저장소 index/ 에 배치해 커밋할 것
print(open("repo/index/manifest.json", encoding="utf-8").read())
from google.colab import files
files.download("repo/index/manifest.json")
files.download("repo/index/build_report.md")
files.download("embed_meta.json")

In [ ]:
# 셀 7 (선택) — 임베딩 원본 회수 (로컬 index/ 캐시 보관용, gitignore 대상, 약 41MB)
from google.colab import files
files.download("embeddings.npy")
files.download("cids.json")